In [46]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
df=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\Feature Engineering\Principal_Component_Analysis\data\ames.csv")
X=df.copy()
y=X.pop("SalePrice")

In [47]:
X=X.fillna(0)
#PCA works only on numeric data
X_num=X.select_dtypes(exclude=['object'])

In [48]:
#Scale Data
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X_num)

In [49]:
#Apply PCA
from sklearn.decomposition import PCA
pca=PCA()
X_pca=pca.fit_transform(X_scaled)
X_pca=pd.DataFrame(
    X_pca,
    columns=[f"PC{i}"for i in range(1,X_pca.shape[1]+1)],
    index=X.index
)

In [50]:
#Mutual Info
#Compute mutual information scores
def make_mi_scores(X,y):
    mi_scores=mutual_info_regression(X,y)
    mi_scores=pd.Series(mi_scores,index=X.columns)
    mi_scores=mi_scores.sort_values(ascending=False)
    return mi_scores



In [51]:
mi_scores=make_mi_scores(X_pca,y)
print(mi_scores.head(10))

PC1     0.884943
PC4     0.106942
PC2     0.099316
PC3     0.082747
PC25    0.064171
PC29    0.056493
PC22    0.056398
PC21    0.053582
PC5     0.051318
PC19    0.049403
dtype: float64


In [52]:
top_pcs=mi_scores.head(5).index

In [53]:
#Join only those pcs back
X_final=X_num.join(X_pca[top_pcs])

In [54]:
#define RMSLE Scoring Function
from sklearn.metrics import mean_squared_log_error
def rmsle(y_true,y_pred):
    return np.sqrt(mean_squared_log_error(y_true,y_pred))

In [55]:
#Cross Validation Scoring Function
def score_dataset(X,y):
    model=RandomForestRegressor(
        n_estimators=50,
        random_state=0,
        n_jobs=-1)
    scores=-cross_val_score(
        model,
        X,
        y,
        cv=3,
        scoring='neg_mean_squared_log_error'
    )
    return np.sqrt(scores.mean())

In [56]:
score=score_dataset(X_final,y)
print("RMSLE : ",score)

RMSLE :  0.16260898102009735


In [57]:
X_base=pd.get_dummies(df.drop('SalePrice',axis=1))
y=df['SalePrice']
score=score_dataset(X_base,y)
print("Base RMSLE : ",score)

Base RMSLE :  0.1514111939489975


In [58]:
X_scaled=scaler.fit_transform(X_base)
pca=PCA(n_components=5)
X_pca=pca.fit_transform(X_scaled)
X_pca=pd.DataFrame(X_pca,
                   columns=[f"PC{i}"for i in range(1,X_pca.shape[1]+1)],index=X_base.index)
score=score_dataset(X_pca,y)
print("Only PCA RMSLE:",score)

Only PCA RMSLE: 0.16486698077119047


In [59]:
X_final=X_base.copy()
X_final=X_final.join(X_pca)
score=score_dataset(X_final,y)
print("Original + PCA RMSLE : ",score)


Original + PCA RMSLE :  0.14029153604288394


In [65]:
print(X_base.shape)
print(X_base.head())


(2930, 348)
   LotFrontage  LotArea  YearBuilt  YearRemodAdd  MasVnrArea  BsmtFinSF1  \
0        141.0  31770.0       1960          1960       112.0         2.0   
1         80.0  11622.0       1961          1961         0.0         6.0   
2         81.0  14267.0       1958          1958       108.0         1.0   
3         93.0  11160.0       1968          1968         0.0         1.0   
4         74.0  13830.0       1997          1998         0.0         3.0   

   BsmtFinSF2  BsmtUnfSF  TotalBsmtSF  FirstFlrSF  ...  SaleType_New  \
0         0.0      441.0       1080.0      1656.0  ...         False   
1       144.0      270.0        882.0       896.0  ...         False   
2         0.0      406.0       1329.0      1329.0  ...         False   
3         0.0     1045.0       2110.0      2110.0  ...         False   
4         0.0      137.0        928.0       928.0  ...         False   

   SaleType_Oth  SaleType_VWD  SaleType_WD   SaleCondition_Abnorml  \
0         False         Fals

In [66]:
print(X_pca.shape)
print(X_pca.head())

(2930, 5)
        PC1       PC2       PC3       PC4       PC5
0  0.132287  3.264499  2.984152  4.094550 -0.507340
1 -3.476156  3.917105 -0.549004 -0.638194 -0.960158
2 -1.380493  4.564432  0.205744  2.032663 -0.297257
3  2.693210  2.467157  1.264607  2.987150 -0.569155
4  2.521581  0.153663 -1.385395 -1.663254  2.934288


In [67]:
print(X_final.shape)
print(X_final.head())

(2930, 353)
   LotFrontage  LotArea  YearBuilt  YearRemodAdd  MasVnrArea  BsmtFinSF1  \
0        141.0  31770.0       1960          1960       112.0         2.0   
1         80.0  11622.0       1961          1961         0.0         6.0   
2         81.0  14267.0       1958          1958       108.0         1.0   
3         93.0  11160.0       1968          1968         0.0         1.0   
4         74.0  13830.0       1997          1998         0.0         3.0   

   BsmtFinSF2  BsmtUnfSF  TotalBsmtSF  FirstFlrSF  ...  SaleCondition_AdjLand  \
0         0.0      441.0       1080.0      1656.0  ...                  False   
1       144.0      270.0        882.0       896.0  ...                  False   
2         0.0      406.0       1329.0      1329.0  ...                  False   
3         0.0     1045.0       2110.0      2110.0  ...                  False   
4         0.0      137.0        928.0       928.0  ...                  False   

   SaleCondition_Alloca  SaleCondition_Famil